# RAG-Trained Weights — Plain-Text Inference (No Retrieval)

This notebook evaluates the RAG-trained classifiers in `weights_rag/` **without any retrieval at inference time**.
The models were fine-tuned on augmented inputs (`query [SEP] neighbor1 [SEP] ...`) but here receive only the raw query text.

**Purpose:** isolate the contribution of training-time augmentation from inference-time retrieval.
If scores are close to the full RAG pipeline, the retrieval at inference time adds little value.
If scores drop, the model relies on neighbors to make good predictions.

**Configurable dimensions (edit Cell 2):**
| Variable | Options |
|---|---|
| `SELECTED_MODELS` | `bert`, `hatebert`, `roberta` |
| `SELECTED_INDEX_TYPES` | `training`, `documents`, `full` |
| `SELECTED_DATASETS` | `IHC`, `ISHate`, `Vicomtech` |

## 1. Imports

In [1]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import warnings
warnings.filterwarnings('ignore')

## 2. Configuration

Edit `SELECTED_MODELS`, `SELECTED_INDEX_TYPES`, and `SELECTED_DATASETS` to choose what to evaluate.

In [2]:
ROOT_DIR        = Path('..')   # notebook lives in evaluation/
WEIGHTS_RAG_DIR = ROOT_DIR / 'weights_rag'

MAX_LENGTH = 256
BATCH_SIZE = 32

# === What to evaluate — edit these lists ===
SELECTED_MODELS      = ['bert', 'roberta']             # 'bert' | 'hatebert' | 'roberta'
SELECTED_INDEX_TYPES = ['training', 'documents', 'full']
SELECTED_DATASETS    = ['IHC', 'ISHate', 'Vicomtech']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device           : {device}')
print(f'Models           : {SELECTED_MODELS}')
print(f'Index types      : {SELECTED_INDEX_TYPES}')
print(f'Datasets         : {SELECTED_DATASETS}')

Device           : cuda
Models           : ['bert', 'roberta']
Index types      : ['training', 'documents', 'full']
Datasets         : ['IHC', 'ISHate', 'Vicomtech']


## 3. Load Datasets

Only datasets in `SELECTED_DATASETS` are loaded.

In [3]:
DATASETS = {}

if 'IHC' in SELECTED_DATASETS:
    raw_ihc = load_dataset('tasksource/implicit-hate-stg1', split='train')
    splits  = raw_ihc.train_test_split(test_size=0.10, seed=42)
    test_ihc = splits['test'].filter(lambda x: x['class'] != 'explicit_hate')
    test_ihc = test_ihc.map(lambda x: {'label': 0 if x['class'] == 'not_hate' else 1})
    DATASETS['IHC'] = {'test': test_ihc, 'text_col': 'post'}
    print(f'IHC       — test: {len(test_ihc):,}')

if 'ISHate' in SELECTED_DATASETS:
    ishate_raw   = load_dataset('BenjaminOcampo/ISHate')
    test_ishate  = ishate_raw['test'].map(lambda x: {'label': 0 if x['hateful_layer'] == 'Non-HS' else 1})
    DATASETS['ISHate'] = {'test': test_ishate, 'text_col': 'text'}
    print(f'ISHate    — test: {len(test_ishate):,}')

if 'Vicomtech' in SELECTED_DATASETS:
    _repo_dir  = str(ROOT_DIR / 'RAG' / 'data' / 'hate-speech-dataset')
    _meta      = pd.read_csv(f'{_repo_dir}/annotations_metadata.csv').set_index('file_id')
    _test_dir  = f'{_repo_dir}/sampled_test'
    _rows = []
    for fname in sorted(os.listdir(_test_dir)):
        if not fname.endswith('.txt'):
            continue
        fid = fname[:-4]
        if fid not in _meta.index:
            continue
        lbl = _meta.loc[fid, 'label']
        if lbl not in ('hate', 'noHate'):
            continue
        with open(os.path.join(_test_dir, fname), encoding='utf-8') as f:
            text = f.read().strip()
        _rows.append({'text': text, 'label': 1 if lbl == 'hate' else 0})
    test_vicomtech = Dataset.from_list(_rows)
    DATASETS['Vicomtech'] = {'test': test_vicomtech, 'text_col': 'text'}
    print(f'Vicomtech — test: {len(test_vicomtech):,}')

README.md:   0%|          | 0.00/792 [00:00<?, ?B/s]

implicit_hate_v1_stg1_posts.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/21480 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2148 [00:00<?, ? examples/s]

Map:   0%|          | 0/2028 [00:00<?, ? examples/s]

...

Generating validation split:   0%|          | 0/4367 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4368 [00:00<?, ? examples/s]

Map:   0%|          | 0/4368 [00:00<?, ? examples/s]

ISHate    — test: 4,368


Vicomtech — test: 478


## 4. Model Registry

Builds all `(model, index_type, dataset)` combinations and checks which weights exist on disk.

In [4]:
MODELS_CONFIG = [
    {
        'path':       WEIGHTS_RAG_DIR / model / 'sbert' / index_type / dataset,
        'label':      f'{model} / sbert / {index_type} / {dataset}',
        'model_name': model,
        'index_type': index_type,
        'dataset':    dataset,
        'text_col':   DATASETS[dataset]['text_col'],
    }
    for model in SELECTED_MODELS
    for index_type in SELECTED_INDEX_TYPES
    for dataset in SELECTED_DATASETS
]

print(f"{'Model / Index / Dataset':<45} Weights?")
print('-' * 55)
for m in MODELS_CONFIG:
    has = (m['path'] / 'model.safetensors').exists() or (m['path'] / 'pytorch_model.bin').exists()
    print(f"{m['label']:<45} {'✓' if has else '✗  (missing)'}")

Model / Index / Dataset                       Weights?
-------------------------------------------------------
bert / sbert / training / IHC                 ✓
bert / sbert / training / ISHate              ✓
bert / sbert / training / Vicomtech           ✓
bert / sbert / documents / IHC                ✓
bert / sbert / documents / ISHate             ✓
bert / sbert / documents / Vicomtech          ✓
bert / sbert / full / IHC                     ✓
bert / sbert / full / ISHate                  ✓
bert / sbert / full / Vicomtech               ✓
roberta / sbert / training / IHC              ✓
roberta / sbert / training / ISHate           ✓
roberta / sbert / training / Vicomtech        ✓
roberta / sbert / documents / IHC             ✓
roberta / sbert / documents / ISHate          ✓
roberta / sbert / documents / Vicomtech       ✓
roberta / sbert / full / IHC                  ✓
roberta / sbert / full / ISHate               ✓
roberta / sbert / full / Vicomtech            ✓


## 5. Helpers

In [5]:
def tokenize_plain(hf_dataset, tokenizer, text_col):
    encoded = tokenizer(
        list(hf_dataset[text_col]),
        truncation=True, padding='max_length', max_length=MAX_LENGTH,
    )
    encoded['labels'] = list(hf_dataset['label'])
    return Dataset.from_dict(encoded)


def compute_metrics(eval_pred):
    preds  = np.argmax(eval_pred.predictions, axis=-1)
    labels = eval_pred.label_ids
    return {
        'macro_f1': f1_score(labels, preds, average='macro',  zero_division=0),
        'macro_p':  precision_score(labels, preds, average='macro', zero_division=0),
        'macro_r':  recall_score(labels, preds, average='macro',    zero_division=0),
    }

## 6. Evaluation Loop

For each available model: tokenize plain text (no retrieval) → predict → store metrics.

In [6]:
results = {}

eval_args = TrainingArguments(
    output_dir='./tmp_eval',
    per_device_eval_batch_size=BATCH_SIZE,
    report_to='none',
)

for entry in MODELS_CONFIG:
    has_weights = (entry['path'] / 'model.safetensors').exists() or (entry['path'] / 'pytorch_model.bin').exists()
    if not has_weights:
        print(f"[skip] {entry['label']} — no weights on disk")
        continue

    print(f"\n{'='*60}")
    print(f"{entry['label']}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(entry['path'])
    test_ds   = DATASETS[entry['dataset']]['test']
    tok_test  = tokenize_plain(test_ds, tokenizer, entry['text_col'])

    model   = AutoModelForSequenceClassification.from_pretrained(entry['path'])
    trainer = Trainer(model=model, args=eval_args, compute_metrics=compute_metrics)

    preds_out = trainer.predict(tok_test)
    preds     = np.argmax(preds_out.predictions, axis=-1)
    labels    = list(test_ds['label'])

    print(classification_report(labels, preds, target_names=['Non-HS', 'HS']))

    results[entry['label']] = {
        'model':      entry['model_name'],
        'index_type': entry['index_type'],
        'dataset':    entry['dataset'],
        'macro_f1':   f1_score(labels, preds, average='macro',  zero_division=0),
        'macro_p':    precision_score(labels, preds, average='macro', zero_division=0),
        'macro_r':    recall_score(labels, preds, average='macro',    zero_division=0),
    }

    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()


bert / sbert / training / IHC


              precision    recall  f1-score   support

      Non-HS       0.80      0.89      0.84      1330
          HS       0.74      0.58      0.65       698

    accuracy                           0.78      2028
   macro avg       0.77      0.74      0.75      2028
weighted avg       0.78      0.78      0.78      2028


bert / sbert / training / ISHate


              precision    recall  f1-score   support

      Non-HS       0.85      0.94      0.89      2681
          HS       0.88      0.75      0.81      1687

    accuracy                           0.86      4368
   macro avg       0.87      0.84      0.85      4368
weighted avg       0.86      0.86      0.86      4368


bert / sbert / training / Vicomtech


              precision    recall  f1-score   support

      Non-HS       0.62      0.96      0.75       239
          HS       0.91      0.42      0.57       239

    accuracy                           0.69       478
   macro avg       0.77      0.69      0.66       478
weighted avg       0.77      0.69      0.66       478


bert / sbert / documents / IHC


              precision    recall  f1-score   support

      Non-HS       0.81      0.89      0.85      1330
          HS       0.74      0.62      0.67       698

    accuracy                           0.79      2028
   macro avg       0.78      0.75      0.76      2028
weighted avg       0.79      0.79      0.79      2028


bert / sbert / documents / ISHate


...

              precision    recall  f1-score   support

      Non-HS       0.92      0.90      0.91      2681
          HS       0.85      0.88      0.86      1687

    accuracy                           0.89      4368
   macro avg       0.88      0.89      0.89      4368
weighted avg       0.89      0.89      0.89      4368


roberta / sbert / documents / Vicomtech


              precision    recall  f1-score   support

      Non-HS       0.75      0.83      0.79       239
          HS       0.81      0.72      0.77       239

    accuracy                           0.78       478
   macro avg       0.78      0.78      0.78       478
weighted avg       0.78      0.78      0.78       478


roberta / sbert / full / IHC


              precision    recall  f1-score   support

      Non-HS       0.83      0.88      0.86      1330
          HS       0.75      0.66      0.70       698

    accuracy                           0.81      2028
   macro avg       0.79      0.77      0.78      2028
weighted avg       0.80      0.81      0.80      2028


roberta / sbert / full / ISHate


              precision    recall  f1-score   support

      Non-HS       0.87      0.94      0.90      2681
          HS       0.89      0.78      0.83      1687

    accuracy                           0.88      4368
   macro avg       0.88      0.86      0.87      4368
weighted avg       0.88      0.88      0.88      4368


roberta / sbert / full / Vicomtech


              precision    recall  f1-score   support

      Non-HS       0.65      0.95      0.77       239
          HS       0.91      0.49      0.64       239

    accuracy                           0.72       478
   macro avg       0.78      0.72      0.71       478
weighted avg       0.78      0.72      0.71       478



## 7. Results — One Table per Dataset

In [7]:
for ds_name in SELECTED_DATASETS:
    ds_results = {
        k: v for k, v in results.items() if v['dataset'] == ds_name
    }
    if not ds_results:
        print(f'No results for {ds_name}\n')
        continue

    rows = {}
    for label, vals in ds_results.items():
        row_key = f"{vals['model']} / sbert / {vals['index_type']}"
        rows[row_key] = {
            'Macro F1':        vals['macro_f1'],
            'Macro Precision': vals['macro_p'],
            'Macro Recall':    vals['macro_r'],
        }

    df = pd.DataFrame(rows).T
    df.index.name = 'Model / Index'

    display(
        df.style
        .format('{:.3f}')
        .highlight_max(axis=0, props='font-weight: bold; background-color: #d4f1d4')
        .set_caption(f'{ds_name} — RAG weights, plain-text inference (no retrieval)')
    )

,Macro F1,Macro Precision,Macro Recall
Model / Index,,,
bert / sbert / training,0.747,0.770,0.736
bert / sbert / documents,0.761,0.777,0.751
bert / sbert / full,0.739,0.758,0.729
roberta / sbert / training,0.777,0.794,0.768
roberta / sbert / documents,0.777,0.786,0.770
roberta / sbert / full,0.780,0.792,0.773


,Macro F1,Macro Precision,Macro Recall
Model / Index,,,
bert / sbert / training,0.851,0.867,0.841
bert / sbert / documents,0.866,0.866,0.865
bert / sbert / full,0.851,0.864,0.843
roberta / sbert / training,0.865,0.881,0.856
roberta / sbert / documents,0.886,0.883,0.888
roberta / sbert / full,0.869,0.882,0.861


,Macro F1,Macro Precision,Macro Recall
Model / Index,,,
bert / sbert / training,0.664,0.766,0.688
bert / sbert / documents,0.739,0.756,0.743
bert / sbert / full,0.591,0.742,0.636
roberta / sbert / training,0.657,0.770,0.684
roberta / sbert / documents,0.778,0.782,0.778
roberta / sbert / full,0.706,0.780,0.722
